In [ ]:
import sys
#print(sys.executable)

In [22]:
#  Imports 
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns

#load the dataset in data/train.csv
data = pd.read_csv('data/train.csv')

In [23]:
#load the dataset in data/train.csv
train_raw = pd.read_csv('data/train.csv')

In [24]:
train_raw.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [25]:
train_raw['MSZoning'].value_counts()

MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64

In [26]:
# LotFrontage imputation via frontage_ratio = LotFrontage / sqrt(LotArea)
# Strategy: median ratio per (Neighborhood, LotConfig) → Neighborhood → global fallback

known = train_raw['LotFrontage'].notna()

# Compute ratio on observed rows
train_raw['_frontage_ratio'] = np.where(
    known,
    train_raw['LotFrontage'] / np.sqrt(train_raw['LotArea']),
    np.nan
)

# Fit median ratios on training data (to be reused on test set later)
ratio_nbhd_cfg = (
    train_raw[known]
    .groupby(['Neighborhood', 'LotConfig'])['_frontage_ratio']
    .median()
)
ratio_nbhd = (
    train_raw[known]
    .groupby('Neighborhood')['_frontage_ratio']
    .median()
)
ratio_global = train_raw.loc[known, '_frontage_ratio'].median()

def impute_frontage(row):
    if pd.notna(row['LotFrontage']):
        return row['LotFrontage']
    ratio = ratio_nbhd_cfg.get(
        (row['Neighborhood'], row['LotConfig']),
        ratio_nbhd.get(row['Neighborhood'], ratio_global)
    )
    return ratio * np.sqrt(row['LotArea'])

train_raw['LotFrontage'] = train_raw.apply(impute_frontage, axis=1)
train_raw.drop(columns=['_frontage_ratio'], inplace=True)

missing_after = train_raw['LotFrontage'].isna().sum()
print(f"Missing LotFrontage after imputation: {missing_after}")
print(f"\nLotFrontage stats after imputation:\n{train_raw['LotFrontage'].describe().round(2)}")


Missing LotFrontage after imputation: 0

LotFrontage stats after imputation:
count    1460.00
mean       70.45
std        25.14
min        21.00
25%        58.00
50%        70.00
75%        81.00
max       313.00
Name: LotFrontage, dtype: float64


In [27]:
import pickle, os

# Save fitted imputer artifacts so the test notebook can apply the same ratios
# without refitting on test data.
os.makedirs('models', exist_ok=True)

lot_frontage_imputer = {
    'ratio_nbhd_cfg': ratio_nbhd_cfg,   # Series keyed by (Neighborhood, LotConfig)
    'ratio_nbhd':     ratio_nbhd,        # Series keyed by Neighborhood
    'ratio_global':   ratio_global,      # scalar fallback
}

with open('models/lot_frontage_imputer.pkl', 'wb') as f:
    pickle.dump(lot_frontage_imputer, f)

print("Saved: models/lot_frontage_imputer.pkl")

# ── HOW TO LOAD IN THE TEST NOTEBOOK ─────────────────────────────────────────
# with open('models/lot_frontage_imputer.pkl', 'rb') as f:
#     imp = pickle.load(f)
# ratio_nbhd_cfg = imp['ratio_nbhd_cfg']
# ratio_nbhd     = imp['ratio_nbhd']
# ratio_global   = imp['ratio_global']
# test['LotFrontage'] = test.apply(impute_frontage, axis=1)
# ─────────────────────────────────────────────────────────────────────────────


Saved: models/lot_frontage_imputer.pkl


In [28]:
train_raw["MasVnrType"].value_counts()

MasVnrType
BrkFace    445
Stone      128
BrkCmn      15
Name: count, dtype: int64

In [29]:
# Investigate MasVnrType missingness vs MasVnrArea
mas_missing = train_raw['MasVnrType'].isna()
print(f"MasVnrType missing: {mas_missing.sum()} rows")
print(f"\nMasVnrArea for rows where MasVnrType is NaN:")
print(train_raw.loc[mas_missing, 'MasVnrArea'].value_counts(dropna=False))
print(f"\nMasVnrType value counts (including NaN):")
print(train_raw['MasVnrType'].value_counts(dropna=False))


MasVnrType missing: 872 rows

MasVnrArea for rows where MasVnrType is NaN:
MasVnrArea
0.0      859
NaN        8
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64

MasVnrType value counts (including NaN):
MasVnrType
NaN        872
BrkFace    445
Stone      128
BrkCmn      15
Name: count, dtype: int64


In [30]:
# MasVnrType imputation
# NaN in this dataset means "None" (no veneer), NOT truly missing.
# Exception: 5 rows have NaN type but MasVnrArea > 0 — impute with most common type.

# 1. Fix MasVnrArea NaN → 0 first (8 rows missing both)
train_raw.loc[train_raw['MasVnrArea'].isna(), 'MasVnrArea'] = 0.0

# 2. NaN type + area == 0 → "None" (no veneer, data encoding artifact)
mask_none = train_raw['MasVnrType'].isna() & (train_raw['MasVnrArea'] == 0)
train_raw.loc[mask_none, 'MasVnrType'] = 'None'

# 3. NaN type + area > 0 → impute with most common type (BrkFace)
mask_has_area = train_raw['MasVnrType'].isna() & (train_raw['MasVnrArea'] > 0)
train_raw.loc[mask_has_area, 'MasVnrType'] = 'BrkFace'

print(f"MasVnrType missing after imputation: {train_raw['MasVnrType'].isna().sum()}")
print(f"MasVnrArea missing after imputation: {train_raw['MasVnrArea'].isna().sum()}")
print(f"\nMasVnrType value counts:\n{train_raw['MasVnrType'].value_counts()}")


MasVnrType missing after imputation: 0
MasVnrArea missing after imputation: 0

MasVnrType value counts:
MasVnrType
None       867
BrkFace    450
Stone      128
BrkCmn      15
Name: count, dtype: int64


In [32]:
train_raw["BsmtFinType1"].value_counts()


BsmtFinType1
Unf    430
GLQ    418
ALQ    220
BLQ    148
Rec    133
LwQ     74
Name: count, dtype: int64

In [34]:
train_raw["BsmtFinType2"].value_counts()

BsmtFinType2
Unf    1256
Rec      54
LwQ      46
BLQ      33
ALQ      19
GLQ      14
Name: count, dtype: int64

In [35]:
# Investigate all basement categorical columns
bsmt_cat = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

print("--- Missing counts ---")
print(train_raw[bsmt_cat + ['BsmtFinSF1', 'TotalBsmtSF']].isna().sum())

# How many of the 5 categorical cols are NaN per row
n_missing = train_raw[bsmt_cat].isna().sum(axis=1)
print("\n--- # of basement cat cols missing per row ---")
print(n_missing.value_counts().sort_index())

# Rows with partial missingness (not all 5 NaN → real basement with a gap)
partial = train_raw[(n_missing > 0) & (n_missing < 5)]
print(f"\n--- Rows with PARTIAL missingness ({len(partial)} rows) ---")
print(partial[bsmt_cat + ['TotalBsmtSF']].to_string())


--- Missing counts ---
BsmtQual        37
BsmtCond        37
BsmtExposure    38
BsmtFinType1    37
BsmtFinType2    38
BsmtFinSF1       0
TotalBsmtSF      0
dtype: int64

--- # of basement cat cols missing per row ---
0    1421
1       2
5      37
Name: count, dtype: int64

--- Rows with PARTIAL missingness (2 rows) ---
    BsmtQual BsmtCond BsmtExposure BsmtFinType1 BsmtFinType2  TotalBsmtSF
332       Gd       TA           No          GLQ          NaN         3206
948       Gd       TA          NaN          Unf          Unf          936


In [36]:
# Basement categorical imputation (all 5 cols in one pass)
# NaN = "no basement" for rows where TotalBsmtSF == 0 (37 rows).
# Remaining isolated NaNs are real basements with one missing field → mode impute.

bsmt_cat = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

# 1. No-basement rows: fill all 5 cols with 'NA'
no_bsmt = train_raw['TotalBsmtSF'] == 0
for col in bsmt_cat:
    train_raw.loc[no_bsmt, col] = 'NA'

# 2. Remaining isolated NaNs: real basement, missing one field → fill with mode
mode_defaults = {
    'BsmtExposure': 'No',   # row 948: basement exists, exposure unknown
    'BsmtFinType2': 'Unf',  # row 332: basement exists, second finish unknown
}
for col, fallback in mode_defaults.items():
    train_raw.loc[train_raw[col].isna(), col] = fallback

print("Missing after imputation:")
print(train_raw[bsmt_cat].isna().sum())


Missing after imputation:
BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
dtype: int64


OJO:  AHORA TENGO QUE SEGUIR CON LOS MISSING FIREPLACE

In [ ]:
# Feature Categorization and Encoding
data = train_raw.copy()

# Drop Id column
data = data.drop('Id', axis=1)

# Ordinal categorical mappings
ordinal_mappings = {
    'OverallQual': {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10},
    'OverallCond': {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10},
    'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtQual': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtCond': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
    'FireplaceQu': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageCond': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'PoolQC': {'NA': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
    'Fence': {'NA': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
}

# Apply ordinal encodings
for col, mapping in ordinal_mappings.items():
    if col in data.columns:
        data[col] = data[col].map(mapping)

# Nominal categorical columns (one-hot encode)
nominal_cols = [
    'MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour',
    'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1',
    'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
    'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating',
    'CentralAir', 'Electrical', 'GarageType', 'PavedDrive', 'MiscFeature',
    'SaleType', 'SaleCondition'
]

# One-hot encode nominal columns (drop_first to avoid multicollinearity)
data = pd.get_dummies(data, columns=nominal_cols, drop_first=True, dtype=int)

print(f"Data shape after categorization: {data.shape}")
print(f"Data dtypes:\n{data.dtypes.value_counts()}")
print(f"\nFirst few rows:\n{data.head()}")